In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path
import re
from string import Template

"""
    Lembrança: rodar novamente as metricas que escolhermos com quantidades variadas de clock para notar a diferença e entender
    Aparentemente, certas metricas podem utilizar o mesmo registrador, oq pode atrabalhar a contagem, talvez seja recomendado ver quais interferem, e retirar da analise
"""

SIZES:tuple[int] = (1000,2000,3000,4000,5000)
FOLDERS_OTIMIZATION_LEVEL:tuple[str] = (("otimizacao-00","00") , ("otimizacao-03", "03"), ("otimizacao-03-native", "03-native"))

metrics_data:dict[str,list]={
          "task_clock":[],
          "cycles":[],
          "instructions":[],
          "branches":[],
          "branch_misses":[],
          "cpu_migrations":[],
          "context_switches":[],
          "cache_references":[],
          "cache_misses":[],
          "major_faults":[],
          "page_faults":[],
          "time_elapsed":[],
          "otimization":[],
          "matrix_size":[]
    }

file_name: Template = Template("${folder}/metrics_otimization${otimization}_${size}_${run}.txt")
for folder, otimization_lv in FOLDERS_OTIMIZATION_LEVEL:
    for size in SIZES:
        for idx in range(1,11):
            file:Path=Path(file_name.substitute(folder=folder, otimization=otimization_lv, size=size, run=idx))
            with file.open(encoding="utf-8") as f:
                conteudo = f.read()
                metrics_data["task_clock"].append(float(re.search(r"([\d.,]+) msec task-clock", conteudo).group(1).replace(".", "").replace(",", ".")))
                
                metrics_data["cycles"].append(int(re.search(r"([\d.,]+)      cycles", conteudo).group(1).replace(".","")))
                
                metrics_data["instructions"].append(int(re.search(r"([\d.,]+)      instructions", conteudo).group(1).replace(".","")))
                
                metrics_data["branches"].append(int(re.search(r"([\d.,]+)      branches", conteudo).group(1).replace(".","")) )
                
                metrics_data["branch_misses"].append(int(re.search(r"([\d.,]+)      branch-misses", conteudo).group(1).replace(".","")))
                
                metrics_data["cpu_migrations"].append(int(re.search(r"([\d.,]+)      cpu-migrations", conteudo).group(1).replace(".","")))
                
                metrics_data["context_switches"].append(int(re.search(r"([\d.,]+)      context-switches", conteudo).group(1).replace(".","")))
                
                metrics_data["cache_references"].append(int(re.search(r"([\d.,]+)      cache-references", conteudo).group(1).replace(".","")))
                
                metrics_data["cache_misses"].append(int(re.search(r"([\d.,]+)      cache-misses", conteudo).group(1).replace(".","")))
                
                metrics_data["major_faults"].append(int(re.search(r"([\d.,]+)      major-faults", conteudo).group(1).replace(".","")))
                
                metrics_data["page_faults"].append( int(re.search(r"([\d.,]+)      page-faults", conteudo).group(1).replace(".","")))
                
                metrics_data["time_elapsed"].append(float(re.search(r"([\d.,]+) seconds time elapsed", conteudo).group(1).replace(".","").replace(",", ".")))

                metrics_data["otimization"].append(otimization_lv)
                
                metrics_data["matrix_size"].append(size)
    
metrics_df = pd.DataFrame(metrics_data)
metrics_df["IPC"] = metrics_df["instructions"] / metrics_df["cycles"] 
#metrics_df.to_csv("metricas_performance_PC_NOVO.csv", sep=";")

In [ ]:
#metrics_df["cache_references"].agg(["mean", "median","max", "min"])
import pandas as pd
import plotly.express as px 

metrics_df = pd.read_csv("metricas_performance_PC_NOVO.csv", sep=";").drop(columns=["Unnamed: 0"])

display(metrics_df.info())
display(metrics_df["cycles"].agg(["mean","median", "std"]))
display(metrics_df.tail())

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   task_clock        150 non-null    float64
 1   cycles            150 non-null    int64  
 2   instructions      150 non-null    int64  
 3   branches          150 non-null    int64  
 4   branch_misses     150 non-null    int64  
 5   cpu_migrations    150 non-null    int64  
 6   context_switches  150 non-null    int64  
 7   cache_references  150 non-null    int64  
 8   cache_misses      150 non-null    int64  
 9   major_faults      150 non-null    int64  
 10  page_faults       150 non-null    int64  
 11  time_elapsed      150 non-null    float64
 12  otimization       150 non-null    str    
 13  matrix_size       150 non-null    int64  
 14  IPC               150 non-null    float64
dtypes: float64(3), int64(11), str(1)
memory usage: 17.7 KB


None

mean      8.842141e+09
median    5.689976e+09
std       9.666229e+09
Name: cycles, dtype: float64

,task_clock,cycles,instructions,branches,branch_misses,cpu_migrations,context_switches,cache_references,cache_misses,major_faults,page_faults,time_elapsed,otimization,matrix_size,IPC
145,2932.72,12918639127,23829197061,5935362000,10371676,0,2258,1160497173,19945577,0,48999,3.042324,03-native,5000,1.844559
146,2907.18,12808006875,23810710194,5933292438,9977058,1,2237,1160784472,19012707,0,48999,3.019218,03-native,5000,1.859049
147,2905.87,12806358457,23828269988,5933486122,10305864,1,2247,1162413070,20690306,0,48999,3.019847,03-native,5000,1.860659
148,2896.54,12757242613,23826082578,5940614584,9689698,1,2242,1165763427,20420497,0,49000,3.009929,03-native,5000,1.867651
149,2915.01,12848374416,23815581203,5937017027,9650821,1,2244,1166763563,23814610,0,49000,3.028112,03-native,5000,1.853587


In [40]:
metrics_O3_native :pd.DataFrame = metrics_df[metrics_df["otimization"]=="03-native"] 

chart1 = px.box(metrics_O3_native, y="branch_misses", x="matrix_size")
display(chart1.show())


metrics_5000mat_df : pd.DataFrame = metrics_df[metrics_df["matrix_size"]==5000].reset_index(drop=True)

chart2 = px.bar(metrics_5000mat_df, y="instructions", color="otimization")
display(chart2.show())


chart3 = px.bar(metrics_df, y="instructions", color="otimization")
display(chart3.show())

None

None

None